# SPINE-GPE v7 — PNADc Historical Certification & Proxy Calibration Engine v1.0.0

Este notebook executa a auditoria, a calibração temporal 2022T4/2024T3 e a certificação das fontes regulares da PNADc já disponíveis no Drive.

**Limite obrigatório:** a série histórica recebe probabilidade modelada (`evidence tier C`); `platform_delivery_direct` permanece ausente.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path('/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7')
SCRIPT = ROOT / 'scripts/SPINE_GPEv7_PNADC_HISTORICAL_PROXY_ENGINE_v1.0.0.py'
REQ = ROOT / 'scripts/requirements_SPINE_GPEv7_PNADC_HISTORICAL_PROXY_ENGINE_v1.0.0.txt'
UPSTREAM = ROOT / 'scripts/SPINE_GPEv7_PNADC_CERTIFIER_v1.2.0.py'

print('ROOT:', ROOT)
print('SCRIPT:', SCRIPT, SCRIPT.exists())
print('REQ:', REQ, REQ.exists())
print('UPSTREAM:', UPSTREAM, UPSTREAM.exists())
assert SCRIPT.exists()
assert REQ.exists()
assert UPSTREAM.exists()


ROOT: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7
SCRIPT: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_PNADC_HISTORICAL_PROXY_ENGINE_v1.0.0.py True
REQ: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/requirements_SPINE_GPEv7_PNADC_HISTORICAL_PROXY_ENGINE_v1.0.0.txt True
UPSTREAM: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_PNADC_CERTIFIER_v1.2.0.py True


In [3]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)], check=True)
subprocess.run([sys.executable, '-m', 'py_compile', str(SCRIPT)], check=True)
print('py_compile: OK')


py_compile: OK


## 1. Auditoria das fontes locais

O modo `auto` descobre os trimestres já materializados em `data_pnadc` e `01_raw/10_ibge/pnadc_historical`, excluindo os módulos especiais diretos.


In [4]:
audit = subprocess.run(
    [
        sys.executable, str(SCRIPT),
        '--root', str(ROOT),
        '--mode', 'audit',
        '--periods', 'auto',
        '--strict',
    ],
    text=True,
    capture_output=True,
    check=False,
)
print(audit.stdout)
print(audit.stderr)
print('Audit exit code:', audit.returncode)


2026-07-21 02:43:16,239 | INFO | SPINE-GPE PNADc Historical Proxy Engine v1.0.0 | root=/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 | mode=audit | periods=auto
2026-07-21 02:43:30,255 | INFO | Auditoria concluída | status=AUDIT_BLOCKED | períodos=['2022q4', '2024q3']
{
  "run_id": "20260721T024316Z",
  "script_version": "1.0.0",
  "validation_schema_version": "spine-gpe-v7-pnadc-historical-validation-1.0.0",
  "mode": "audit",
  "status": "AUDIT_BLOCKED",
  "critical_failures": [
    {
      "test_id": "historical.2022q4.layout",
      "status": "FAIL",
      "severity": "critical",
      "message": "Falha de layout/source kind: Nenhum layout direto casou com a largura 3478 para 2022. Candidatos: [{'score': 220, 'path': '/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/02_interim/10_pnadc_certification/extracted_docs/Dicionario_e_input_20221031/dicionario_PNADC_microdados_trimestral.xls', 'width': 3478, 'has_S140093': False}, {'score': 220, 'path': '/content/drive/MyDrive

In [5]:
AUDIT_LOCK = ROOT / '00_admin/PNADC_HISTORICAL_PROXY_AUDIT_LOCK.json'
if not AUDIT_LOCK.exists():
    raise RuntimeError('Lock de auditoria não foi criado. Revise STDOUT/STDERR.')
audit_lock = json.loads(AUDIT_LOCK.read_text(encoding='utf-8'))
print(json.dumps(audit_lock, ensure_ascii=False, indent=2))
assert audit_lock['status'] == 'AUDIT_PASSED', audit_lock['critical_failures']
print('Períodos descobertos:', audit_lock['requested_periods'])


{
  "run_id": "20260721T024316Z",
  "script_version": "1.0.0",
  "validation_schema_version": "spine-gpe-v7-pnadc-historical-validation-1.0.0",
  "mode": "audit",
  "status": "AUDIT_BLOCKED",
  "critical_failures": [
    {
      "test_id": "historical.2022q4.layout",
      "status": "FAIL",
      "severity": "critical",
      "message": "Falha de layout/source kind: Nenhum layout direto casou com a largura 3478 para 2022. Candidatos: [{'score': 220, 'path': '/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/02_interim/10_pnadc_certification/extracted_docs/Dicionario_e_input_20221031/dicionario_PNADC_microdados_trimestral.xls', 'width': 3478, 'has_S140093': False}, {'score': 220, 'path': '/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/02_interim/10_pnadc_certification/extracted_docs/Dicionario_e_input_20221031/input_PNADC_trimestral.sas', 'width': 3478, 'has_S140093': False}, {'score': 220, 'path': '/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/02_interim/10_pnadc_cer

AssertionError: [{'test_id': 'historical.2022q4.layout', 'status': 'FAIL', 'severity': 'critical', 'message': "Falha de layout/source kind: Nenhum layout direto casou com a largura 3478 para 2022. Candidatos: [{'score': 220, 'path': '/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/02_interim/10_pnadc_certification/extracted_docs/Dicionario_e_input_20221031/dicionario_PNADC_microdados_trimestral.xls', 'width': 3478, 'has_S140093': False}, {'score': 220, 'path': '/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/02_interim/10_pnadc_certification/extracted_docs/Dicionario_e_input_20221031/input_PNADC_trimestral.sas', 'width': 3478, 'has_S140093': False}, {'score': 220, 'path': '/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/02_interim/10_pnadc_certification/extracted_docs/Dicionario_e_input_20221031/input_PNADC_trimestral.txt', 'width': 3478, 'has_S140093': False}, {'score': 220, 'path': '/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/02_interim/pnadc_documentation

## 2. Calibração e certificação

Esta etapa:

1. lê os Parquets diretos certificados de 2022 e 2024;
2. valida temporalmente o modelo nos dois sentidos;
3. calibra a probabilidade;
4. aplica o modelo apenas aos trimestres regulares descobertos;
5. gera outputs imutáveis, model card, estimativas, lock e freeze.


In [6]:
full = subprocess.run(
    [
        sys.executable, str(SCRIPT),
        '--root', str(ROOT),
        '--mode', 'full',
        '--periods', 'auto',
        '--chunk-rows', '50000',
        '--strict',
    ],
    text=True,
    capture_output=True,
    check=False,
)
print(full.stdout)
print(full.stderr)
print('Full exit code:', full.returncode)


KeyboardInterrupt: 

In [ ]:
LOCK = ROOT / '00_admin/PNADC_HISTORICAL_PROXY_CERTIFICATION_LOCK.json'
if not LOCK.exists():
    raise RuntimeError('Lock de certificação não foi criado. Revise STDOUT/STDERR.')
lock = json.loads(LOCK.read_text(encoding='utf-8'))
print(json.dumps(lock, ensure_ascii=False, indent=2))
assert lock['status'] == 'CERTIFIED', lock['critical_failures']
print('STATUS:', lock['status'])
print('MODELO:', lock['model'])
print('PERÍODOS:', lock['certified_periods'])
print('REPORT:', lock['report'])


## 3. Inspeção das métricas temporais e estimativas


In [ ]:
import pandas as pd

run_id = lock['run_id']
metrics_path = ROOT / f'05_outputs/tables/pnadc_historical_proxy/pnadc_proxy_temporal_metrics_{run_id}.csv'
rules_path = ROOT / f'05_outputs/tables/pnadc_historical_proxy/pnadc_proxy_rule_benchmark_{run_id}.csv'
estimates_path = Path(lock['estimates'])

metrics = pd.read_csv(metrics_path)
rules = pd.read_csv(rules_path)
estimates = pd.read_csv(estimates_path)

display(metrics)
display(rules)
display(estimates)


## 4. Expansão histórica opcional

Execute somente depois que o modo `auto` estiver certificado. O comando abaixo pode baixar e processar muitos arquivos grandes.


In [ ]:
# Exemplo controlado: período pandêmico e pré-módulo direto.
# Remova o comentário para executar.
# expansion = subprocess.run(
#     [
#         sys.executable, str(SCRIPT),
#         '--root', str(ROOT),
#         '--mode', 'full',
#         '--periods', '2019q1:2021q4',
#         '--download-missing',
#         '--chunk-rows', '50000',
#         '--strict',
#     ],
#     check=False,
# )


## Regras de uso

- A soma ponderada das probabilidades é o estimando principal.
- A classe binária é somente análise de sensibilidade.
- A série histórica não preenche `SD14001`, `S140093` ou `platform_delivery_direct`.
- Comparações entre períodos são descritivas/model-based, não um desenho causal.
